# Compute mean change confidence intervals

Full temporal coverage: compute change confidence intervals for every index configured in `configs/change_ci.yaml`, using all same-month consecutive-year pairs from 2019 to 2025 across all study areas. Results are saved under `outputs/change_ci/<INDEX>/`.

In [ ]:
from pathlib import Path

import pandas as pd
import rasterio

from change_ci import AnalysisConfig, ChangeCIAnalysis, build_pairs, discover_rasters

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"

## Analysis configuration

In [ ]:
config = AnalysisConfig.from_yaml(
    ROOT / "configs" / "change_ci.yaml", project_root=ROOT
)
config.to_dict()

## Discover all same-month consecutive-year pairs

Expected coverage: 168 date pairs per index (seven study areas × four months × six year transitions).

In [ ]:
raster_records = discover_rasters(config.data_root)
missing_bands = []
for record in raster_records:
    with rasterio.open(record.path) as raster:
        missing = set(config.index_bands) - set(raster.descriptions)
    if missing:
        missing_bands.append((record.path, sorted(missing)))
if missing_bands:
    raise ValueError(f'Configured index bands missing from composites: {missing_bands[:10]}')
pair_specs = build_pairs(raster_records, config)
study_areas = config.study_areas or tuple(sorted({record.study_area for record in raster_records}))
expected_pairs = len(study_areas) * len(config.months) * (config.year_range[1] - config.year_range[0])
if len(pair_specs) != expected_pairs:
    raise ValueError(f'Expected {expected_pairs} date pairs, discovered {len(pair_specs)}')
pair_table = pd.DataFrame([
    {
        'study_area': pair.study_area,
        'month': pair.month,
        'year_start': pair.year_start,
        'year_end': pair.year_end,
        'earlier_path': pair.earlier_path,
        'later_path': pair.later_path,
    }
    for pair in pair_specs
])
pair_table

## Run and save

In [ ]:
analysis = ChangeCIAnalysis(config)
results, correlograms = analysis.run(pair_specs)
analysis.save(results, correlograms)
print(f'Saved {len(results)} result rows and {len(correlograms)} correlogram rows under {config.output_dir}')
index_summary = results.groupby('index').agg(
    result_rows=('index', 'size'),
    failed_pairs=('error', lambda values: values.fillna('').ne('').sum()),
)
display(index_summary)
expected_rows = len(pair_specs) * len(config.tau_sensitivity)
index_rows = index_summary['result_rows'].reindex(config.index_bands, fill_value=0)
if not index_rows.eq(expected_rows).all() or index_summary['failed_pairs'].sum():
    raise RuntimeError('At least one index has incomplete or failed pair results; inspect saved result tables.')

## Results and review flags

In [ ]:
primary = results.loc[results['is_primary_tau']].copy()
summary_columns = [
    'study_area', 'index', 'month', 'year_start', 'year_end', 'n', 'mean_change', 'sd',
    'range_px', 'range_map_units', 'n_eff', 'ci_low', 'ci_high',
    'naive_ci_low', 'naive_ci_high', 'flag_no_range', 'flag_low_n_eff', 'error',
]
display(primary[summary_columns].sort_values(['index', 'study_area', 'month', 'year_start']))

review = primary.loc[
    primary['flag_no_range']
    | primary['flag_low_n_eff']
    | primary['error'].fillna('').ne('')
]
print(f'{len(review)} primary-threshold result(s) flagged or failed.')
display(review[summary_columns].sort_values(['index', 'study_area', 'month', 'year_start']))